# 対話システムの基礎（ルールベースと検索型）

書籍 第4章「言語生成問題の解き方」（4-3 対話：チャットのやり取りをする）に対応する演習ノートブックです。
CPU のみ・数秒で実行でき、生成モデル（LLM）以前の対話システムの基本設計である次の 2 つのアプローチを体験します。

1. **ルールベース（テンプレート / ELIZA 風パターンマッチ）**（4-3-1）
2. **用例検索型（TF-IDF 類似発話検索）**（4-3-2）
3. **入力の揺らぎ・未知発話に対するフォールバックとキャラクター設計の比較**

## 1. 準備

In [1]:
import re
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

---

## 2. ルールベース対話システム（4-3-1）

書籍 4-3-1 で紹介されている **ELIZA**（Weizenbaum, 1966）のように、正規表現で入力文のキーワードやパターンを捉え、あらかじめ用意したテンプレートに当てはめて応答を返すボットを作ります。
ここでは「研究室の先輩チャットボット（丁寧・相談役キャラ）」を想定してみます。

In [2]:
rules = [
    (r"(こんにちは|こんばんは|はじめまして|おはよう)",
     lambda m: "こんにちは！自然言語処理の勉強や研究のことで何か相談はありますか？"),
    (r"(.+)(が|は)(難しい|わからない|理解できない)",
     lambda m: f"「{m.group(1)}」は最初は難しく感じますよね。どのあたりで詰まっていますか？"),
    (r"(BERT|Transformer|GPT|LLM)",
     lambda m: f"{m.group(1)} は強力ですが、まずは TF-IDF やルールベースでベースラインを作るのもおすすめですよ。"),
    (r"(データ|アノテーション|コーパス)",
     lambda m: "データの質はモデル以上に大事です！仕様書を作って複数人で一致率を測ってみましたか？"),
    (r"(ありがとう|感謝|助かり)",
     lambda m: "どういたしまして！またいつでも声をかけてくださいね。"),
]

def rule_based_reply(user_text):
    for pattern, template_fn in rules:
        m = re.search(pattern, user_text)
        if m:
            return template_fn(m)
    return "なるほど、もう少し詳しく教えていただけますか？"

test_inputs = [
    "こんにちは！",
    "Transformerが難しいです",
    "アノテーションの進め方に迷っています",
    "今日はいい天気ですね",  # ルールに一致しない発話（フォールバック）
    "ありがとうございます！",
]

for u in test_inputs:
    print(f"ユーザ: {u}")
    print(f"ボット: {rule_based_reply(u)}\n")

ユーザ: こんにちは！
ボット: こんにちは！自然言語処理の勉強や研究のことで何か相談はありますか？

ユーザ: Transformerが難しいです
ボット: 「Transformer」は最初は難しく感じますよね。どのあたりで詰まっていますか？

ユーザ: アノテーションの進め方に迷っています
ボット: データの質はモデル以上に大事です！仕様書を作って複数人で一致率を測ってみましたか？

ユーザ: 今日はいい天気ですね
ボット: なるほど、もう少し詳しく教えていただけますか？

ユーザ: ありがとうございます！
ボット: どういたしまして！またいつでも声をかけてくださいね。



ルールベースは**想定したシナリオ内では破綻せずキャラクターの一貫性も保てる**一方、少し言い回しが変わったり想定外の発話が来たりすると画一的なフォールバック応答しか返せません。

---

## 3. 用例検索型（Retrieval-based）対話システム（4-3-2）

次に、書籍 4-3-2 の用例検索ベースとして、過去の Q&A・対話ペアのデータベースから「ユーザの入力に最も意味（表層）が近い質問」を TF-IDF コサイン類似度で検索し、対応する応答を返すシステムを作ります。
実運用では「類似度が一定の閾値未満なら応答せずフォールバックする」設計が重要です。

In [3]:
faq_pairs = [
    ("GPUがなくて学習が重いです", "まずは CPU で動く TF-IDF や線形モデルでベースラインを作り、その後 Google Colab の無料 T4 GPU を使うのがおすすめです。"),
    ("評価指標は何を使えばよいですか", "分類なら正解率に加えて F1、生成タスクなら BLEU や ROUGE、品質推定なら相関係数や Kendall's tau などをタスクに合わせて選びます。"),
    ("教師データがなかなか集まりません", "少量のデータにアノテーションして一致率を確認したり、ルールベースや事前学習モデルのゼロショットから始める手があります。"),
    ("形態素解析ツールはどれがおすすめですか", "Python から手軽に使うなら SudachiPy や fugashi（MeCab）、Web テキストの文分割なら Bunkai などがよく使われます。"),
    ("要約モデルを作る前に何をすべきですか", "ニュース記事などでは先頭数文を抜き出す「リード法」が強力なベースラインになるので、まずリード法と ROUGE で比較しましょう。"),
    ("翻訳モデルの精度を測るにはどうしますか", "参照訳がある場合は sacrebleu で BLEU を測り、参照訳がない場合は COMET-QE などの品質推定モデルが使えます。"),
]

questions = [q for q, _ in faq_pairs]
answers = [a for _, a in faq_pairs]

vec = TfidfVectorizer(analyzer="char_wb", ngram_range=(1, 3))
q_mat = vec.fit_transform(questions)

def retrieval_reply(user_text, threshold=0.25):
    u_vec = vec.transform([user_text])
    sims = cosine_similarity(u_vec, q_mat).ravel()
    best_idx = int(np.argmax(sims))
    best_sim = float(sims[best_idx])
    if best_sim < threshold:
        return best_sim, None, "すみません、その質問に合う回答が見つかりませんでした。別の言い方で質問してみてください。"
    return best_sim, questions[best_idx], answers[best_idx]

queries = [
    "Google ColabでGPUが使えないときはどうしたらいい？",
    "要約のベースライン手法を教えてください",
    "形態素解析のおすすめライブラリは？",
    "明日の東京の天気を教えてください",  # ドメイン外の質問（閾値で弾く）
]

for q in queries:
    sim, matched_q, ans = retrieval_reply(q)
    print(f"ユーザ: {q}")
    print(f"  [類似度={sim:.3f}, ヒット='{matched_q}']")
    print(f"ボット: {ans}\n")

ユーザ: Google ColabでGPUが使えないときはどうしたらいい？
  [類似度=0.417, ヒット='GPUがなくて学習が重いです']
ボット: まずは CPU で動く TF-IDF や線形モデルでベースラインを作り、その後 Google Colab の無料 T4 GPU を使うのがおすすめです。

ユーザ: 要約のベースライン手法を教えてください
  [類似度=0.278, ヒット='要約モデルを作る前に何をすべきですか']
ボット: ニュース記事などでは先頭数文を抜き出す「リード法」が強力なベースラインになるので、まずリード法と ROUGE で比較しましょう。

ユーザ: 形態素解析のおすすめライブラリは？
  [類似度=0.676, ヒット='形態素解析ツールはどれがおすすめですか']
ボット: Python から手軽に使うなら SudachiPy や fugashi（MeCab）、Web テキストの文分割なら Bunkai などがよく使われます。

ユーザ: 明日の東京の天気を教えてください
  [類似度=0.173, ヒット='None']
ボット: すみません、その質問に合う回答が見つかりませんでした。別の言い方で質問してみてください。



### まとめ（4-3）

- **ルールベース**: 想定範囲の会話やキャラクター維持に強く、誤った情報を生成するリスクがありません。
- **検索ベース**: 言い回しの揺らぎに一定程度対応でき、人手が書いた自然な応答を返せます（閾値によるフォールバック設計が重要）。
- **生成ベース（Seq2Seq / LLM）**: 未知の質問にも柔軟に応答を生成できますが、ハルシネーションやキャラ崩壊を防ぐための制御や評価が課題になります。